# CardiaSense — Model tiers, training & the benchmark gate

**Educational technical walkthrough · notebook 3 of 5**

**Objectives:** classical baseline, deep smoke training, campaign tables and the adopt/skip gate

**You will need:** the `cardiasense` venv (run `./setup.sh`), the dataset at
`$CARDIASENSE_DATA_DIR` (downloaded by `setup.sh`), and a trained head for the
notebooks that use checkpoints (or run the smoke train in notebook 03 first).

> Educational / research prototype — **not a medical device**.

---




### Step 1 — Classical baseline (fast, interpretable)

**Features:** 13 MFCC coefficients + their Δ and Δ² (temporal dynamics), plus
spectral centroid / bandwidth, zero-crossing rate, RMS and spectral flatness —
aggregated to mean/std per clip.

**Models:** logistic regression (LR), random forest (RF), and a **calibrated**
SVM. All use `class_weight="balanced"` to counter the imbalance. Models fit on
the **train split only** (val is reserved for model selection — same protocol as
the deep stages), scored on the held-out test fold.

**Expected:** the classical baseline is the *gate champion* on every task — a
cheap, strong reference the deep tiers must beat by ≥ 1 balanced-accuracy point.


In [1]:

import os, sys
from pathlib import Path

ROOT = Path("/home/tnzr/CardiaSense-AI")
sys.path.insert(0, str(ROOT / "ml"))
sys.path.insert(0, str(ROOT))

from cardia.config import DEFAULT_DATA_DIR   # loads .env if present
DATA = os.environ.get("CARDIASENSE_DATA_DIR") or DEFAULT_DATA_DIR
print("Dataset root:", DATA)


Dataset root: /home/tnzr/CardiaSense-AI/data/HLS-CMDS


In [2]:

from cardia.config import Config
from cardia.data.hls_cmds import build_task_manifest
from cardia.data.splits import make_splits
from cardia.train import classical_train
import wandb, pandas as pd

cfg = Config(task="heart", stage=1, out_dir=str(ROOT / "ml" / "runs"))
m = build_task_manifest(DATA, "heart", "binary")
splits = make_splits(m, n_folds=5, seed=0)
run = wandb.init(mode="disabled")
metrics = classical_train.run_classical(cfg, m, splits, ["normal", "abnormal"], run, "nb-walkthrough")
run.finish()
print(pd.DataFrame(metrics["_per_classifier"])[["classifier", "balanced_accuracy_mean", "macro_f1_mean"]].to_string(index=False))


/home/tnzr/CardiaSense-AI/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   classifier  balanced_accuracy_mean  macro_f1_mean
     logistic                0.886008       0.844785
random_forest                0.882143       0.915634
          svm                0.909202       0.921345



### Step 2 — Deep tier (smoke mode) via the CLI

The deep stages (stage 2 CNN → stage 3 temporal → stage 4 transformer) are
invoked through `cardia.cli`. `--smoke` cuts to 2 folds × 2 epochs so you can
see the whole path (dataloaders, class weights, cosine LR, early stopping,
checkpoint, metrics) in about a minute on GPU.

**What it prints:** val balanced accuracy / macro-F1 per epoch, then the
held-out test metrics with a confusion matrix, ECE and latency.


In [3]:

import subprocess
rc = subprocess.run([sys.executable, "-m", "cardia.cli", "--task", "heart", "--stage", "2",
                     "--model", "resnet18", "--fold", "0", "--smoke", "--out-dir", "/tmp/cardiasense_nb_runs"],
                    cwd=str(ROOT / "ml"))
print("CLI exit code:", rc.returncode)


wandb: [wandb.Api()] Loaded credentials for https://api.wandb.ai from /home/tnzr/.netrc.


wandb: Currently logged in as: tnzr (tnzr-pioneer-innovations-collective) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Tracking run with wandb version 0.30.0
wandb: Run data is saved locally in /tmp/cardiasense_nb_runs/wandb/wandb/run-20261010_220956-jltwiggq
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run stage2-heart-resnet18-allfolds-s0
wandb: ⭐️ View project at https://wandb.ai/tnzr-pioneer-innovations-collective/cardiasense-hls-cmds
wandb: 🚀 View run at https://wandb.ai/tnzr-pioneer-innovations-collective/cardiasense-hls-cmds/runs/jltwiggq


wandb: WARNING Artifact "stage2-heart-resnet18-allfolds-s0-fold0_best" already exists with the same content. No new version will be created.


wandb: updating run metadata; uploading artifact run-jltwiggq-outputpredictions-Y75LRA; uploading artifact run-jltwiggq-fold_results; uploading artifact run-jltwiggq-cvconfusion_matrix--CFlKQ


wandb: uploading artifact run-jltwiggq-fold_results; uploading artifact run-jltwiggq-cvconfusion_matrix--CFlKQ


wandb: uploading media/images/cv/confusion_matrix_fig_27_ae190343e4b1b9b693d5.png; uploading media/table/cv/confusion_matrix_28_719336bf8165f15ab7df.table.json; uploading output.log; uploading wandb-summary.json; uploading config.yaml (+ 3 more)


wandb: 
wandb: Run history:
wandb:                 epoch ▁█▁█
wandb:                    lr █▁█▁
wandb:            train/loss ▃▇█▁
wandb:          val/accuracy ▃▆█▁
wandb:             val/auroc █▁▇▃
wandb: val/balanced_accuracy ▁▁▁▁
wandb:              val/loss █▄▆▁
wandb:          val/macro_f1 ▄▇█▁
wandb: 
wandb: Run summary:
wandb:                best_epoch 0
wandb:             cv/best_epoch 0
wandb:  cv/latency_ms_per_sample 5.53136
wandb:          cv/test_accuracy 0.49942
wandb:             cv/test_auroc 0.49242
wandb: cv/test_balanced_accuracy 0.5
wandb:               cv/test_ece 0.50058
wandb:          cv/test_macro_f1 0.28543
wandb:   cv/test_macro_precision 0.24971
wandb:      cv/test_macro_recall 0.5
wandb:                       +18 ...
wandb: 
wandb: 🚀 View run stage2-heart-resnet18-allfolds-s0 at: https://wandb.ai/tnzr-pioneer-innovations-collective/cardiasense-hls-cmds/runs/jltwiggq
wandb: ⭐️ View project at: https://wandb.ai/tnzr-pioneer-innovations-collective/cardiasense-h

[cardia] task=heart manifest=195 samples
{
  "task": "heart",
  "stage": 2,
  "model": "resnet18",
  "fold": null,
  "seed": 0,
  "test_accuracy_mean": 0.49942141805175677,
  "test_accuracy_std": 0.547550285747336,
  "test_balanced_accuracy_mean": 0.5,
  "test_balanced_accuracy_std": 0.0,
  "test_macro_f1_mean": 0.2854313931919587,
  "test_macro_f1_std": 0.26094214750660133,
  "test_macro_precision_mean": 0.24971070902587839,
  "test_macro_precision_std": 0.273775142873668,
  "test_macro_recall_mean": 0.5,
  "test_macro_recall_std": 0.0,
  "test_auroc": 0.49242424242424243,
  "test_ece_mean": 0.5005785819482432,
  "test_ece_std": 0.547550285747336,
  "val_balanced_accuracy_mean": 0.5,
  "val_balanced_accuracy_std": 0.0,
  "latency_ms_per_sample": 5.531359631972719,
  "best_epoch": 0
}


CLI exit code: 0



### Step 3 — The campaign & the gate rule

`scripts/run_sequential.sh` runs all four stages for every task, then applies
the **gate**: a stage is adopted only if balanced accuracy **or** macro-F1
improves by ≥ 1 point over the best prior stage. Both metrics are logged, so the
decision is auditable in `ml/runs/metrics.json` and the benchmark table.


In [4]:

import json
p = ROOT / "ml" / "runs" / "benchmark.json"
if p.exists():
    bench = json.loads(p.read_text())["benchmark"]
    print(pd.DataFrame(bench).to_string(index=False))
else:
    print("No benchmark.json yet — run:  cd ml && SMOKE=1 ./scripts/run_sequential.sh")


 task  stage  best_test_balanced_accuracy  best_test_macro_f1 gate_decision  best_prior_balanced_accuracy  best_prior_macro_f1
sound      1                     0.934159            0.935068         adopt                           NaN                  NaN
sound      2                     0.938167            0.939672          skip                      0.934159             0.935068
sound      3                     0.851105            0.856311          skip                      0.934159             0.935068
sound      4                     0.860242            0.863494          skip                      0.934159             0.935068
heart      1                     0.909202            0.921345         adopt                           NaN                  NaN
heart      2                     0.652521            0.535970          skip                      0.909202             0.921345
heart      3                     0.668235            0.454096          skip                      0.909202      